# Model 2: price forecast + landed costs — run notebook
Run top to bottom. Every cell is safe to re-run. Needs `model2.py` next to this notebook and Model 1's `model1.py` (found automatically).

1. **Setup**: paths, imports, finds `model1.py` and the team submission
2. **Load + data checks**: futures check (doc's test) and landed-cost formula fit
3. **Fill `price_forecast`** in the team submission (only that column changes)
4. **Submission check**: the doc's pre-upload checklist, all columns
5. **Landed costs for Model 3**: `outputs/landed_costs.csv`
6. **Results**: backtest, example forecasts, ablation chart (PNGs for the video)
7. **Phase 2**: does nothing until weeks after 208 are in `Data/`

## 1. Setup

In [0]:
import os
import sys
import glob
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", None)

# repo root = first folder above this notebook that contains Data/
d = os.getcwd()
while d != os.path.dirname(d) and not os.path.isdir(os.path.join(d, "Data")):
    d = os.path.dirname(d)
REPO_ROOT = d if os.path.isdir(os.path.join(d, "Data")) else "/Workspace/Repos/DSDG Hackathon/DSDG Hackathon"
DATA = os.path.join(REPO_ROOT, "Data")

# Model 1: use iter3 if it's there, otherwise the most recently changed model1.py outside archive folders
MODEL1_DIR = os.path.join(REPO_ROOT, "Code Base", "pipelining", "Model 1", "iter3")
if not os.path.exists(os.path.join(MODEL1_DIR, "model1.py")):
    found = [p for p in glob.glob(os.path.join(REPO_ROOT, "**", "model1.py"), recursive=True) if "archive" not in p]
    MODEL1_DIR = os.path.dirname(max(found, key=os.path.getmtime))
if MODEL1_DIR not in sys.path:
    sys.path.insert(0, MODEL1_DIR)
import model1
import model2

# one shared submission file: Model 1's notebook creates it, each model fills only its own columns
TEAM_SUBMISSION = os.path.join(MODEL1_DIR, "outputs", "submission.csv")
OUT = os.path.join(os.getcwd(), "outputs")
os.makedirs(OUT, exist_ok=True)

print("DATA:           ", DATA)
print("model1.py from: ", MODEL1_DIR)
print("team submission:", TEAM_SUBMISSION, "(exists)" if os.path.exists(TEAM_SUBMISSION) else "(not yet - run Model 1's notebook first)")
print("OUT:            ", OUT)

## 2. Load data and check the two assumptions the model rests on
- **Futures check (doc's test):** do futures at T predict spot at T+4 better than spot at T? Expect *no* in calm weeks, *yes* only when the premium is large (pre-shock).
- **Landed cost formula:** `landed = spot x (1 + tariff) + b x container_rate + c` per supplier. Expect max error under 1%.

In [0]:
pdata = model2.PriceData(model1.load_data(DATA))
P = pdata.P
PHASE2 = P.last_week > 208
print(f"data runs through week {P.last_week}; Phase 2 loaded: {PHASE2}\n")
print(model2.futures_check(pdata).round(4).to_string(index=False))

lf = model2.landed_fit_report(pdata)
print(f"\nlanded-cost formula: mean error {lf.mape.mean():.4%}, worst {lf.max_error.max():.3%}")
assert lf.max_error.max() < 0.02, "landed-cost formula no longer fits - check supply_history / tariffs / shipping"
lf.round(5)

## 3. Fill `price_forecast`
Reads the team submission (Model 1's `disruption_prob` stays), fills `price_forecast`, writes it back, and keeps a copy in this folder's `outputs/`.
Before Phase 2 data, weeks 222-234 are forecast from week 208.

In [0]:
sub_in = TEAM_SUBMISSION if os.path.exists(TEAM_SUBMISSION) else os.path.join(DATA, "sample_submission.csv")
with warnings.catch_warnings():
    warnings.simplefilter("ignore", UserWarning)
    sub = model2.fill_submission(pdata, sub_in)

if os.path.exists(TEAM_SUBMISSION):
    sub.to_csv(TEAM_SUBMISSION, index=False)
sub.to_csv(os.path.join(OUT, "submission.csv"), index=False)
print("read: ", sub_in)
print("wrote:", TEAM_SUBMISSION if os.path.exists(TEAM_SUBMISSION) else "", os.path.join(OUT, "submission.csv"))
if not PHASE2:
    print("note: weeks 222-234 use the week-208 origin until Phase 2 data is loaded")

wk = sub.id.str.extract(r"^w(\d+)_(.+)$")
prices = sub.assign(week=wk[0].astype(int), material=wk[1])
prices.pivot(index="week", columns="material", values="price_forecast").round(3)

## 4. Submission check (all columns)

In [0]:
def check_submission(sub, suppliers):
    issues = []
    if len(sub) != 260:
        issues.append(f"{len(sub)} rows, expected 260")
    if sub.id.duplicated().any():
        issues.append("duplicate ids")
    if sub.isna().any().any():
        issues.append(f"missing values in {sub.columns[sub.isna().any()].tolist()}")
    if not sub.disruption_prob.between(0, 1).all():
        issues.append("disruption_prob outside [0, 1]")
    if not (sub.price_forecast > 0).all():
        issues.append("price_forecast <= 0")
    if (sub.order_units < 0).any() or not np.allclose(sub.order_units, sub.order_units.round()):
        issues.append("order_units negative or not whole numbers")
    share_cols = [c for c in sub.columns if c.startswith("share_")]
    if not np.allclose(sub[share_cols].sum(axis=1), 1.0, atol=1e-6):
        issues.append("shares don't sum to 1 on every row")
    mat = sub.id.str.extract(r"^w\d+_(.+)$")[0]
    sells = suppliers.groupby("material").country.apply(set)
    for col in share_cols:
        cty = col.split("_", 1)[1]
        bad = (sub[col] > 0) & ~mat.map(lambda m: cty in sells[m])
        if bad.any():
            issues.append(f"{col} > 0 for materials {cty} doesn't sell: {sorted(mat[bad].unique())}")
    return issues

problems = check_submission(sub, P.suppliers)
print("READY TO UPLOAD" if not problems else "FIX BEFORE UPLOAD:\n- " + "\n- ".join(problems))
same_as_sample = (sub.disruption_prob == pd.read_csv(os.path.join(DATA, "sample_submission.csv")).disruption_prob).all()
if same_as_sample:
    print("note: disruption_prob is still the sample's - run Model 1's notebook so its column is in the team file")

## 5. Landed costs for Model 3
One row per supplier per week, for each window (each from its own forecast origin). Which column to use:
- `landed_cost_expected`: probability-weighted with Model 1's shock odds. **Quick LP** (one forecast + safety buffer).
- `landed_cost_calm`: no shocks at all (calm price level, base tariff, normal container rate). **Scenario LP**, which applies its own shock cost multipliers; using the expected column there counts shocks twice.
- `landed_cost`: the single most likely path (same assumptions as `price_forecast`). For reporting.
- `spot_*`, `tariff_*`, `container_*`: the pieces, if Model 3 wants to rebuild cost under its own scenarios.

In [0]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore", UserWarning)
    lc = model2.export_for_model3(pdata, out_path=os.path.join(OUT, "landed_costs.csv"))
print("wrote", os.path.join(OUT, "landed_costs.csv"), f"({len(lc)} rows)")
summary = lc.groupby(["window", "material", "country"]).agg(
    landed_expected=("landed_cost_expected", "mean"), landed_calm=("landed_cost_calm", "mean"),
    tariff_max=("tariff_expected", "max"))
summary["cheapest_in_material"] = summary.groupby(level=[0, 1]).landed_expected.transform("min") == summary.landed_expected
summary.round(3)

## 6. Results
Rolling backtest: forecast from every 4th week (104-204), 1-13 weeks ahead, using only data known at the time (Model 1 is refit at each origin too).
Skill = 1 - MAPE / MAPE of carry-forward (0 = carry-forward, the competition's baseline).

In [0]:
bt = model2.backtest(pdata)
print(bt.summary().round(4).to_string())
by_h = bt.by("h")
by_mat = bt.by("material")

fig, ax = plt.subplots(1, 3, figsize=(17, 4))
ax[0].bar(by_h.index, by_h.skill, color="#4C72B0")
ax[0].axhline(0, color="black", lw=0.8)
ax[0].set(title="MAPE skill vs carry-forward, by weeks ahead", xlabel="weeks ahead (h)", ylabel="skill")

def example(axis, material, origin, title):
    mi = P.materials.index(material)
    w = np.arange(origin - 12, min(origin + 14, P.last_week + 1))
    axis.plot(w, pdata.bench[mi, [P.idx(x) for x in w]], color="black", lw=1.5, label="actual")
    b, _ = model2.PriceModel().fit(pdata, origin=origin).predict()
    b = b[b.material == material]
    axis.plot(b.week, b.price_forecast, color="#DD8452", lw=2, label="model 2")
    axis.hlines(pdata.bench[mi, P.idx(origin)], origin + 1, origin + 13, color="grey", ls="--", label="carry-forward")
    axis.axvline(origin, color="grey", lw=0.8)
    axis.set(title=title, xlabel="week", ylabel="benchmark price")
    axis.legend(fontsize=8)

example(ax[1], "copper", 134, "Copper, forecast from week 134 (in a shock)")
example(ax[2], "steel", 156, "Steel, forecast from week 156 (shock warning)")
plt.tight_layout()
fig.savefig(os.path.join(OUT, "model2_backtest.png"), dpi=150)
plt.show()
print(by_mat.round(4).sort_values("skill").to_string())

**Ablation ladder** (doc's versions plus the two jump-rule alternatives). Re-runs the backtest 7 times; loads the saved table unless `RERUN_ABLATION = True`.

In [0]:
RERUN_ABLATION = False
ab_path = os.path.join(OUT, "ablation_model2.csv")
if RERUN_ABLATION or not os.path.exists(ab_path):
    ab = model2.ablation(pdata)
    ab.to_csv(ab_path)
else:
    ab = pd.read_csv(ab_path, index_col=0)

cols = ["overall", "calm_at_origin", "in_shock_at_origin", "h1-4", "h5-8", "h9-13", "doc_origins", "mape", "landed_mape_expected"]
print(ab[[c for c in cols if c in ab.columns]].round(4).to_string())

used = [n for n in ab.index if "submitted" in n]
colors = ["#DD8452" if n in used else "#bbbbbb" for n in ab.index]
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.barh(ab.index[::-1], ab.overall[::-1], color=colors[::-1])
for i, v in enumerate(ab.overall[::-1]):
    ax.text(max(v, 0) + 0.002, i, f"{v:.3f}", va="center", ha="left")
lo, hi = min(0, ab.overall.min()) * 1.6, ab.overall.max() * 1.25
ax.set(title="Model 2 ablation: MAPE skill vs carry-forward (orange = submitted)", xlabel="skill (0 = carry-forward)", xlim=(lo, hi))
ax.axvline(0, color="black", lw=0.8)
plt.tight_layout()
fig.savefig(os.path.join(OUT, "ablation_model2.png"), dpi=150)
plt.show()

## 7. Phase 2
Does nothing until `Data/` has weeks after 208 (append the new weeks with the cell in Model 1's notebook first). Then it refills `price_forecast` from week 221, re-exports landed costs, and re-scores. Run Model 1's notebook before this one so the team file has the new `disruption_prob`.

In [0]:
pdata = model2.PriceData(model1.load_data(DATA))
P = pdata.P
if P.last_week <= 208:
    print(f"No Phase 2 data yet (data ends at week {P.last_week}). Nothing to do.")
else:
    print(model2.futures_check(pdata).round(4).to_string(index=False))
    prem = pdata.fut[:, P.idx(209):] / pdata.spot[:, P.idx(209):] - 1
    big = np.argwhere(prem > 0.05)
    print("\nsupplier futures premiums above 5% since week 209 (market pricing a shock):")
    for u, t in big:
        print(f"  {P.suppliers.material[u]}/{P.suppliers.country[u]} week {209 + t}: {prem[u, t]:.1%}")
    sub_in = TEAM_SUBMISSION if os.path.exists(TEAM_SUBMISSION) else os.path.join(DATA, "sample_submission.csv")
    sub = model2.fill_submission(pdata, sub_in)
    if os.path.exists(TEAM_SUBMISSION):
        sub.to_csv(TEAM_SUBMISSION, index=False)
    sub.to_csv(os.path.join(OUT, "submission.csv"), index=False)
    model2.export_for_model3(pdata, out_path=os.path.join(OUT, "landed_costs.csv"))
    print(model2.backtest(pdata).summary().round(4).to_string())
    print("refilled price_forecast and landed_costs.csv from origin week", min(221, P.last_week))